In [1]:
# ===============================
# MOUNT DRIVE
# ===============================
from google.colab import drive
drive.mount('/content/drive')

# ===============================
# IMPORTS
# ===============================
import cv2
import numpy as np
import os
import pandas as pd
import time
import tracemalloc
import platform
import multiprocessing

from scipy.special import gamma

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error as mse
from skimage.measure import shannon_entropy

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

np.random.seed(42)

# ===============================
# SYSTEM INFO
# ===============================
print("===================================")
print("SYSTEM CONFIGURATION")
print("===================================")
print("Processor:", platform.processor())
print("CPU Cores:", multiprocessing.cpu_count())
print("Platform:", platform.platform())

# ===============================
# LOAD CNN MODEL
# ===============================
cnn_model = VGG16(
    weights='imagenet',
    include_top=False
)

# ===============================
# PATHS
# ===============================
image_folder = "/content/drive/MyDrive/Colab Notebooks/data/low_quality_images"

output_ics = "/content/drive/MyDrive/Colab Notebooks/results/CNN/ICS_CNN"
output_hybrid = "/content/drive/MyDrive/Colab Notebooks/results/CNN/HYBRID_CNN"

os.makedirs(output_ics, exist_ok=True)
os.makedirs(output_hybrid, exist_ok=True)

# ===============================
# METRICS
# ===============================
def compute_metrics(original, processed):

    original = original.astype(np.float32)
    processed = processed.astype(np.float32)

    ssim_val = ssim(
        original,
        processed,
        data_range=255
    )

    psnr_val = psnr(
        original,
        processed,
        data_range=255
    )

    mse_val = mse(
        original,
        processed
    )

    mae_val = np.mean(
        np.abs(original - processed)
    )

    entropy_val = shannon_entropy(
        processed
    )

    edges_orig = cv2.Canny(
        original.astype(np.uint8),
        100,
        200
    )

    edges_proc = cv2.Canny(
        processed.astype(np.uint8),
        100,
        200
    )

    if np.sum(edges_orig) == 0:
        epi_val = 0
    else:
        epi_val = (
            np.sum(edges_orig & edges_proc)
            /
            np.sum(edges_orig)
        )

    return (
        ssim_val,
        psnr_val,
        mse_val,
        mae_val,
        entropy_val,
        epi_val
    )

# ===============================
# RUNTIME EFFICIENCY
# ===============================
def runtime_efficiency(runtime):
    if runtime == 0:
        return 0
    return 1 / runtime

# ===============================
# FAST CNN FEATURE EXTRACTION
# ===============================
def extract_features(img):

    img = cv2.resize(
        img,
        (128,128)
    )

    img = np.stack(
        [img]*3,
        axis=-1
    )

    img = np.expand_dims(
        img,
        axis=0
    )

    img = preprocess_input(
        img.astype(np.float32)
    )

    features = cnn_model.predict(
        img,
        verbose=0
    )

    return features.flatten()

# ===============================
# TRANSFORM
# ===============================
def apply_transform(y, a, b):

    y = y.astype(np.float32)/255.0

    out = 1/(1+np.exp(-a*(y-b)))

    return np.clip(
        out*255,
        0,
        255
    ).astype(np.uint8)

# ===============================
# EDGE METRIC
# ===============================
def tenengrad(img):

    gx = cv2.Sobel(
        img,
        cv2.CV_64F,
        1,
        0,
        ksize=3
    )

    gy = cv2.Sobel(
        img,
        cv2.CV_64F,
        0,
        1,
        ksize=3
    )

    return np.mean(
        gx**2 + gy**2
    )

# ===============================
# FITNESS FUNCTION
# ===============================
def get_fitness(img):

    original_features = extract_features(img)

    def f(p):

        transformed = apply_transform(
            img,
            p[0],
            p[1]
        )

        transformed_features = extract_features(
            transformed
        )

        feature_loss = np.linalg.norm(
            original_features
            -
            transformed_features
        )

        edge = tenengrad(
            transformed
        )

        return (
            -(0.6 * feature_loss)
            +
            (0.4 * edge)
        )

    return f

# ===============================
# ICS CLASS
# ===============================
class ICS:

    def __init__(
        self,
        fitness,
        bounds,
        n=8,
        pa=0.25,
        beta=1.5,
        iters=6
    ):

        self.fit = fitness
        self.bounds = np.array(bounds)

        self.n = n
        self.pa = pa
        self.beta = beta
        self.iters = iters

        self.dim = len(bounds)

        self.pop = np.random.uniform(
            self.bounds[:,0],
            self.bounds[:,1],
            (n,self.dim)
        )

        self.fvals = np.array([
            self.fit(x)
            for x in self.pop
        ])

    def levy(self):

        b = self.beta

        sigma = (
            gamma(1+b)
            *
            np.sin(np.pi*b/2)
            /
            (
                gamma((1+b)/2)
                *
                b
                *
                2**((b-1)/2)
            )
        )**(1/b)

        u = np.random.normal(
            0,
            sigma,
            self.dim
        )

        v = np.random.normal(
            0,
            1,
            self.dim
        )

        return u/(np.abs(v)**(1/b))

    def clip(self,x):

        return np.clip(
            x,
            self.bounds[:,0],
            self.bounds[:,1]
        )

    def run(self):

        for _ in range(self.iters):

            for i in range(self.n):

                new = self.clip(
                    self.pop[i]
                    +
                    0.01*self.levy()
                )

                fnew = self.fit(new)

                j = np.random.randint(
                    self.n
                )

                if fnew > self.fvals[j]:

                    self.pop[j] = new
                    self.fvals[j] = fnew

            for i in range(self.n):

                if np.random.rand() < self.pa:

                    self.pop[i] = np.random.uniform(
                        self.bounds[:,0],
                        self.bounds[:,1],
                        self.dim
                    )

                    self.fvals[i] = self.fit(
                        self.pop[i]
                    )

        return self.pop[
            np.argmax(self.fvals)
        ]

# ===============================
# LOAD IMAGES
# ===============================
files = [
    f for f in os.listdir(image_folder)
    if f.lower().endswith(
        (".jpg",".png",".jpeg")
    )
][:25]

records = []

bounds = [
    (0.5,4.0),
    (0.2,0.8)
]

# ===============================
# MAIN LOOP
# ===============================
for file in files:

    img = cv2.imread(
        os.path.join(
            image_folder,
            file
        )
    )

    gray = cv2.resize(
        cv2.cvtColor(
            img,
            cv2.COLOR_BGR2GRAY
        ),
        (256,256)
    )

    # ==========================
    # ICS CNN
    # ==========================
    tracemalloc.start()

    start_ics = time.perf_counter()

    ics = ICS(
        get_fitness(gray),
        bounds
    )

    best = ics.run()

    enhanced = apply_transform(
        gray,
        best[0],
        best[1]
    )

    end_ics = time.perf_counter()

    current_mem, peak_mem = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    ics_time = end_ics - start_ics
    ics_memory = peak_mem / (1024*1024)
    ics_eff = runtime_efficiency(
        ics_time
    )

    cv2.imwrite(
        os.path.join(
            output_ics,
            file
        ),
        enhanced
    )

    # ==========================
    # HYBRID CNN
    # ==========================
    tracemalloc.start()

    start_hyb = time.perf_counter()

    pre = cv2.GaussianBlur(
        gray,
        (5,5),
        0
    )

    ics_pre = ICS(
        get_fitness(pre),
        bounds
    )

    best_pre = ics_pre.run()

    hybrid = apply_transform(
        pre,
        best_pre[0],
        best_pre[1]
    )

    end_hyb = time.perf_counter()

    current_mem, peak_mem = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    hyb_time = end_hyb - start_hyb
    hyb_memory = peak_mem / (1024*1024)
    hyb_eff = runtime_efficiency(
        hyb_time
    )

    cv2.imwrite(
        os.path.join(
            output_hybrid,
            file
        ),
        hybrid
    )

    # ==========================
    # QUALITY METRICS
    # ==========================
    ics_vals = compute_metrics(
        gray,
        enhanced
    )

    hyb_vals = compute_metrics(
        gray,
        hybrid
    )

    records.append([
        file,

        *ics_vals,
        ics_time,
        ics_memory,
        ics_eff,

        *hyb_vals,
        hyb_time,
        hyb_memory,
        hyb_eff
    ])

    print(
        f"Processed: {file}"
    )

# ===============================
# SAVE RESULTS
# ===============================
df = pd.DataFrame(
    records,
    columns=[

        "Image",

        "ICS_SSIM",
        "ICS_PSNR",
        "ICS_MSE",
        "ICS_MAE",
        "ICS_Entropy",
        "ICS_EPI",
        "ICS_Time",
        "ICS_Memory_MB",
        "ICS_Runtime_Eff",

        "HYBRID_SSIM",
        "HYBRID_PSNR",
        "HYBRID_MSE",
        "HYBRID_MAE",
        "HYBRID_Entropy",
        "HYBRID_EPI",
        "HYBRID_Time",
        "HYBRID_Memory_MB",
        "HYBRID_Runtime_Eff"
    ]
)

df.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/results/CNN/results.csv",
    index=False
)

# ===============================
# PERFORMANCE SUMMARY
# ===============================
performance_summary = pd.DataFrame({

    "Method":[
        "ICS_CNN",
        "HYBRID_CNN"
    ],

    "Average_Time_sec":[
        df["ICS_Time"].mean(),
        df["HYBRID_Time"].mean()
    ],

    "Average_Memory_MB":[
        df["ICS_Memory_MB"].mean(),
        df["HYBRID_Memory_MB"].mean()
    ],

    "Average_Runtime_Efficiency":[
        df["ICS_Runtime_Eff"].mean(),
        df["HYBRID_Runtime_Eff"].mean()
    ]
})

performance_summary.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/results/CNN/performance_summary.csv",
    index=False
)

print("\n===================================")
print("AVERAGE IMAGE QUALITY METRICS")
print("===================================")
print(df.mean(numeric_only=True))

print("\n===================================")
print("COMPUTATIONAL PERFORMANCE")
print("===================================")
print(performance_summary)

print("\nFINAL RUN DONE 🚀")

Mounted at /content/drive
SYSTEM CONFIGURATION
Processor: x86_64
CPU Cores: 2
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Processed: 00 (74).jpg
Processed: 00 (139).jpg
Processed: 00 (84).jpg
Processed: 00 (137).jpg
Processed: 00 (59).jpg
Processed: 00 (112).jpg
Processed: 00 (79).jpg
Processed: 00 (57).jpg
Processed: 00 (132).jpg
Processed: 00 (81).jpg
Processed: 00 (111).jpg
Processed: 00 (109).jpg
Processed: 00 (106).jpg
Processed: 00 (116).jpg
Processed: 00 (131).jpg
Processed: 00 (78).jpg
Processed: 00 (77).jpg
Processed: 00 (141).jpg
Processed: 00 (138).jpg
Processed: 00 (80).jpg
Processed: 00 (54).jpg
Processed: 00 (113).jpg
Processed: 00 (134).jpg
Processed: 00 (133).jpg
Processed: 00 (107).jpg

AVERAGE IMAGE QUALITY METRICS
ICS_SSIM                0.940681
ICS_PSNR               22.317932
ICS_MSE               704.277999
ICS_MAE                21.136059
ICS_Entropy             7.109641
ICS_EPI                 0.851375
ICS_T